# Rédacteur IA sourcé

Agent LangChain qui recherche l'actualité dans une sélection de journaux (via Tavily) et rédige un article de blog en français, avec ses sources.

**Avant de lancer** : ajoutez `OPENAI_API_KEY` et `TAVILY_API_KEY` dans les Secrets Colab (icône 🔑), puis *Exécution → Tout exécuter*. Les paramètres à modifier sont dans la section « Paramètres ».

# Installation des librairies

In [ ]:
!pip install -q "langchain>=1.0,<2" "langchain-openai>=1.0,<2" "langgraph>=1.0,<2" "tavily-python>=0.7,<1" rich markdown

# Variables d'environnement
Les clés sont lues depuis les Secrets Colab (icône 🔑 à gauche).

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY").strip()

# Paramètres : journaux, mots-clés et sujet de l'article

In [ ]:
JOURNAUX = [
    "https://www.elwatan.com",
    "https://www.tsa-algerie.com",
    "https://www.elkhabar.com",
    "https://www.echoroukonline.com",
    "https://www.ennaharonline.com/fr",
    "https://www.lematindalgerie.com",
    "https://echaab.dz",
]

MOTS_CLES = [
    "statut autoentrepreneur", "statut auto-entrepreneur", "chomage",
    "travail", "jeunes", "carte auto-entrepreneur",
]

SUJET = """le nouveau statut d'auto-entrepreneur en Algérie : article de loi, date de parution,
nombre de détenteurs de la carte d'auto-entrepreneur, toute information utile,
et les avantages pour les jeunes et pour l'Algérie"""

# Import des modules

In [ ]:
from urllib.parse import urlparse
from tavily import TavilyClient
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# Outil Tavily : recherche limitée aux journaux choisis

In [ ]:
tavily = TavilyClient()

DOMAINES = [urlparse(u).netloc.replace("www.", "") for u in JOURNAUX]

@tool
def analyse_presse(keywords: str) -> str:
    """Recherche des articles de presse algérienne sur les mots-clés fournis.
    Renvoie pour chaque article son titre, son URL et un extrait."""
    result = tavily.search(
        query=f"Algérie {keywords}",
        include_domains=DOMAINES,
        max_results=8,
    )
    items = result.get("results", [])
    if not items:
        return "Aucun résultat trouvé."
    return "\n\n".join(
        f"TITRE : {r['title']}\nURL : {r['url']}\nEXTRAIT : {r['content']}"
        for r in items
    )

# Création de l'agent
Les consignes de rédaction sont dans `SYSTEM_PROMPT`, séparées de la demande.

In [ ]:
SYSTEM_PROMPT = """Tu es rédacteur web pour un blog algérien grand public.

Méthode :
- Utilise l'outil analyse_presse plusieurs fois, avec des mots-clés différents, pour couvrir le sujet.
- N'utilise que les faits trouvés par l'outil. Si un chiffre ou une date n'apparaît pas dans les résultats,
  ne l'invente pas : laisse-le de côté.

Langue :
- Écris tout l'article en français, sans aucun mot ni phrase en arabe, même si certaines sources sont en arabe :
  traduis les informations que tu en tires.
- Dans la section Sources, traduis en français le titre des articles en arabe et ajoute « (article en arabe) » après le lien.

Style :
- Article attrayant, vivant, facile à lire, avec un titre accrocheur, une introduction courte et des intertitres.
- Écris les faits directement. N'écris jamais « selon la presse », « d'après les médias »,
  « des témoignages relayés par la presse » ou formules équivalentes.

Format de sortie :
- Renvoie UNIQUEMENT l'article, en Markdown. Pas de synthèse avant, pas de proposition ou de question après.
- Termine par une section « ## Sources » listant les articles utilisés sous forme de liens Markdown
  [Titre](URL), en reprenant exactement les URL renvoyées par l'outil. N'invente aucune URL."""

model = ChatOpenAI(model="gpt-5.1")

agent = create_agent(
    model=model,
    tools=[analyse_presse],
    system_prompt=SYSTEM_PROMPT,
)

# Exécution de l'agent

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"Mots-clés de départ : {', '.join(MOTS_CLES)}.\n\nRédige un article de blog sur {SUJET}.",
    }]
})

article = result["messages"][-1].content

# Vérifier les sources consultées
Liste les URL que Tavily a réellement renvoyées à l'agent.

In [ ]:
import re

n = 0
for msg in result["messages"]:
    if msg.type == "tool":
        n += 1
        urls = re.findall(r"URL : (\S+)", msg.content)
        print(f"--- Recherche de l'agent n°{n} ---")
        for u in urls:
            print(u)
        if not urls:
            print(msg.content[:200])
        print()

print(f"Total : {n} recherche(s)")

# Afficher et enregistrer l'article
Génère `article.md` et `article.html`. Pour WordPress, utilise le `.html`.

In [ ]:
import markdown
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from google.colab import files

Console().print(Panel(Markdown(article), title="📰 Article généré", border_style="cyan"))

# Titre (ligne "# ...") séparé du corps, pour le champ titre de WordPress
lignes = article.strip().splitlines()
titre = lignes[0].lstrip("# ").strip() if lignes and lignes[0].startswith("# ") else ""
corps = "\n".join(lignes[1:] if titre else lignes).strip()

html = markdown.markdown(corps)
html = html.replace("<a href=", '<a target="_blank" rel="noopener" href=')

with open("article.md", "w", encoding="utf-8") as f:
    f.write(article)
with open("article.html", "w", encoding="utf-8") as f:
    f.write(html)

print("TITRE À COLLER DANS WORDPRESS :", titre)
files.download("article.html")
files.download("article.md")